# Model experiment  

Try GAT and GCN implementation 

In [30]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import GCNConv, GATv2Conv

# Define data loader
import sys
import gc
sys.path.append('../')
from model.data_loader import PEFDataset,fetch_dataloader
from model.data_loader import params as data_params
from model.model_cfg import CFG

from tqdm import tqdm

In [31]:
def criterion(E,X_native,X_decoy,model,N,h):
    """
    The loss function for the model coressponds to 3 main losses:
    1. lossg: the partial derivateve of the energy with respect to the native structure
    2. lossd: the energy of the native structure divided by the decoy energy
    3. lossc: After preforming an iterative optimization on the decoy structure, by using the energy partial derivative on the decoy structure, 
              we calculate the dRMSD of the end and the start of the optimization.

    Args:
        E (tensor): A tensor containing the energy of the native and the decoy structure Exd,Exn [batch_size*3,2]
        X_native (tensor): A tensor containing the native structure [batch_size,seq_len,4,3]
        X_decoy (tensor): A tensor containing the decoy structure [batch_size,seq_len,4,3]
        model (torch.model): model that was trained
        N (int): The number of iterations for the iterative optimization
        h (float): The step size for the numerical derivative
    output:
        loss (tensor): The loss of the model
    """
    # print('***Start criterion function***')
    # partial_dx_decoy = torch.autograd.grad(E[:,0].sum(),model.parameters(),create_graph=True,allow_unused=True)
    partial_dx_native = torch.autograd.grad(E[:,1].sum(),model.parameters(),create_graph=True,allow_unused=True)
    # print('***End derivative calc function***')
    
    part_dx_native = [ 0 if part_dx is None else torch.norm(part_dx,p=2) for part_dx in partial_dx_native]
    lossg = sum(part_dx_native)
    
    lossd = (E[:,1] / E[:,0]).mean()
    
    # lossc = preform_energy_optimization(X_decoy,partial_dx_decoy)
    
    return (lossd+lossg)

In [32]:
class GCN(torch.nn.Module):
  """Graph Convolutional Network"""
  def __init__(self, dim_in, dim_h, dim_out):
    super().__init__()
    self.gcn1 = GCNConv(dim_in, dim_h)
    self.gcn2 = GCNConv(dim_h, dim_out)
    self.optimizer = torch.optim.Adam(self.parameters(),
                                      lr=0.01,
                                      weight_decay=5e-4)

  def forward(self, x, edge_index):
    h = F.dropout(x, p=0.5, training=self.training)
    h = self.gcn1(h, edge_index)
    h = torch.relu(h)
    h = F.dropout(h, p=0.5, training=self.training)
    h = self.gcn2(h, edge_index)
    return h, F.log_softmax(h, dim=1)


class GAT(torch.nn.Module):
  """Graph Attention Network"""
  def __init__(self, dim_in, dim_h, dim_out, heads=8):
    super().__init__()
    self.gat1 = GATv2Conv(dim_in, dim_h, heads=heads)
    self.gat2 = GATv2Conv(dim_h*heads, dim_out, heads=1)
    self.optimizer = torch.optim.Adam(self.parameters(),
                                      lr=0.005,
                                      weight_decay=5e-4)

  def forward(self, x, edge_index):
    h = F.dropout(x, p=0.6, training=self.training)
    h = self.gat1(x, edge_index)
    h = F.elu(h)
    h = F.dropout(h, p=0.6, training=self.training)
    h = self.gat2(h, edge_index)
    return h, F.log_softmax(h, dim=1)



In [42]:
# define one epoch train
def training (model, optimizer, dataloader, device,N):
    """
    Training function for the model.
    Args:
        model (torch.model): model to train
        optimizer (torch.optim): optimizer to use
        dataloader (torch.utils.data.DataLoader): dataloader for the training set
        device (torch.device): device to use ('cpu' or 'cuda' or 'mps')
        N (int): The number of iterations for the iterative optimization
    """
    model.train()
    
    for epoch in range(CFG.num_epochs):  # loop over the dataset multiple times

        running_loss = 0.0
        torch.cuda.empty_cache()
        gc.collect()
        with tqdm(dataloader, unit="batch") as tepoch:
            for i, data in enumerate(tepoch):
                # set progress bar description
                tepoch.set_description(f"Epoch {epoch}")
                # Clean the GPU cache
                torch.cuda.empty_cache()
                gc.collect()
                # get the inputs; data is a list of [inputs, labels]   
                seq_one_hot,seq_decoy ,id, Xd,Xn, mask, nativemask, esm_embed = data
                Xd = Xd.to(device)
                Xn = Xn.to(device)
                esm_embed = esm_embed.to(device)
                seq_one_hot = seq_one_hot.to(device) # [batch_size,20,seq_len]
                seq_one_hot = torch.swapaxes(seq_one_hot,1,2) # swap the axes to [batch_size,seq_len,20]
                seq_decoy = torch.swapaxes(seq_decoy,1,2)
                #emb = torch.cat((esm_embed,seq),dim=2)
                emb = seq_one_hot
                emb_decoy = seq_decoy.to(device)
                # zero the parameter gradients
                optimizer.zero_grad()
                Xd = Xd.squeeze()
                Xd = Xd.reshape(Xd.shape[0],-1)
                emb_decoy = emb_decoy.squeeze()
                
                Xd_features = torch.cat((Xd,emb_decoy),dim=1)
                print(Xd_features.shape)
                # create edge_index
                edge_index = torch.tensor([],dtype=torch.long)
                # forward + backward + optimize
                for i in range(Xd_features.shape[0]):
                    for j in range(i,Xd_features.shape[0]):
                        if i == j:
                            continue
                        else:
                            edge_index = torch.cat((edge_index,torch.tensor([[i,j]],dtype=torch.long)),dim=0) 
                edge_index = edge_index.to(device)
                print(edge_index.shape)
                outputs = model(Xd_features,edge_index.t().contiguous())
                loss = criterion(outputs,Xd,Xn,model,N,CFG.h)
                print(loss.item())

                loss.backward()
                # print_par(model) # print the parameters of the model
                optimizer.step()

                # print statistics
                running_loss += loss.item()
                if i % 2000 == 1999:    # print every 2000 mini-batches
                    print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
                    running_loss = 0.0
                
                torch.cuda.empty_cache()
                gc.collect()
                # update the progress bar
                tepoch.set_postfix(loss=round(loss.item(),3))
                # save the model
                # 
                # save_checkpoint(epoch, model, optimizer,loss,CFG.model_path)
    print('Finished Training')

In [34]:
# fetch data
print('***load the data with dataloader***')
d_params = data_params(num_workers =CFG.num_workers, batch_size=CFG.batch_size,cuda=CFG.cuda,debug=CFG.debug)
train_loader, valid_loader,test_loader = fetch_dataloader(data_dir='.'+CFG.data_path, params=d_params)


***load the data with dataloader***
Checking data constrain...


100%|██████████| 80/80 [00:01<00:00, 42.35it/s]


Checking data constrain...


100%|██████████| 10/10 [00:00<00:00, 38.12it/s]


Checking data constrain...


100%|██████████| 10/10 [00:00<00:00, 48.54it/s]


In [43]:
# Build the model
print('***Build the model***')
model = GAT(dim_in=32,dim_h=64,dim_out=36).to(CFG.device)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
# Run training
print('***Start training***')
training(model, optimizer, train_loader, CFG.device,CFG.N)

***Build the model***
***Start training***


Epoch 0:   0%|          | 0/18 [00:03<?, ?batch/s]

torch.Size([302, 32])
torch.Size([45451, 2])


Epoch 0:   0%|          | 0/18 [00:14<?, ?batch/s]


RuntimeError: src.device().is_cpu() INTERNAL ASSERT FAILED at "csrc/cpu/scatter_cpu.cpp":11, please report a bug to PyTorch. src must be CPU tensor

In [ ]:
torch.ones(2,2)

tensor([[1., 1.],
        [1., 1.]])